# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/boddulavinaykumar6-stack/Flyrank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Baseline rule

Pages with high search visibility but relatively poor average search position should be prioritized for SEO review. These pages already receive impressions, so improving their rankings may increase organic traffic.

### Reason code

- `high_visibility_low_rank`

### Action label

- `seo_review`

### Signal check 1 — Search impressions

The first signal is **Google Search impressions**. Pages with many impressions already have search demand, making them stronger candidates for optimization.

**Verdict:** **CONFIRMED**

The bucket counts show that pages with high impressions are relatively uncommon, supporting the use of impressions as one signal for prioritization.

### Signal check 2 — Average search position

The second signal is **average search position**. Pages that rank outside the top search results may have greater potential for improvement.

**Verdict:** **CONFIRMED**

The bucket counts show a substantial number of pages in positions 10 and beyond, supporting the use of average position as a ranking-quality signal.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip -q install huggingface_hub duckdb pyarrow pandas

from google.colab import userdata
from huggingface_hub import login
import duckdb
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [2]:
signal1 = con.sql(f"""
SELECT
    CASE
        WHEN gsc_impressions < 100 THEN '<100'
        WHEN gsc_impressions < 1000 THEN '100–999'
        WHEN gsc_impressions < 5000 THEN '1000–4999'
        ELSE '5000+'
    END AS impression_bucket,
    COUNT(*) AS n
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
GROUP BY impression_bucket
ORDER BY
    CASE impression_bucket
        WHEN '<100' THEN 1
        WHEN '100–999' THEN 2
        WHEN '1000–4999' THEN 3
        ELSE 4
    END
""").df()

display(signal1)
print("Total rows:", signal1["n"].sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impression_bucket,n
0,<100,2972453
1,100–999,606189
2,1000–4999,31676
3,5000+,743


Total rows: 3611061


In [3]:
signal2 = con.sql(f"""
SELECT
    CASE
        WHEN gsc_avg_position < 10 THEN '1–9'
        WHEN gsc_avg_position < 20 THEN '10–19'
        WHEN gsc_avg_position < 50 THEN '20–49'
        ELSE '50+'
    END AS position_bucket,
    COUNT(*) AS n
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
GROUP BY position_bucket
ORDER BY
    CASE position_bucket
        WHEN '1–9' THEN 1
        WHEN '10–19' THEN 2
        WHEN '20–49' THEN 3
        ELSE 4
    END
""").df()

display(signal2)
print("Total rows:", signal2["n"].sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n
0,1–9,2154941
1,10–19,543782
2,20–49,633131
3,50+,279207


Total rows: 3611061


## 2. Build the ranked queue

The baseline score prioritizes pages that already receive search visibility but have relatively poor average search positions.

The rule is intentionally simple and transparent:

- Higher impressions increase priority because they indicate existing search demand.
- Worse average search positions increase priority because these pages have more room for ranking improvements.
- Every selected page receives the same reason code (`high_visibility_low_rank`) and action label (`seo_review`).

This score is intended as a transparent baseline that can be compared with a machine learning model in later weeks.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

baseline = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    AVG(gsc_impressions) AS avg_impressions,
    AVG(gsc_avg_position) AS avg_position,

    AVG(gsc_impressions) * AVG(gsc_avg_position) AS score,

    'high_visibility_low_rank' AS reason_code,

    'seo_review' AS action_label

FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE
    gsc_data_available IS TRUE
    AND gsc_impressions > 0
    AND gsc_avg_position > 0

GROUP BY
    client_hash_id,
    content_hash_id

ORDER BY score DESC
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [9]:
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

baseline.to_csv(
    output_path,
    index=False
)

print(f"Wrote {len(baseline):,} rows")
print(f"Saved to: {output_path}")

print(f"\nTotal ranked pages: {len(baseline):,}")
display(baseline.head(10))

Wrote 175,304 rows
Saved to: work/outputs/baseline_action_score.csv

Total ranked pages: 175,304


,client_hash_id,content_hash_id,avg_impressions,avg_position,score,reason_code,action_label
0,client_23a62021009f63c4,content_36e53e9c707674fc,6276.741935,32.766674,205667.959346,high_visibility_low_rank,seo_review
1,client_23a62021009f63c4,content_e8a52cf3d5988c07,7901.000000,15.008339,118580.884331,high_visibility_low_rank,seo_review
2,client_23a62021009f63c4,content_73aa61dcedebbf30,2584.645161,45.700431,118119.397155,high_visibility_low_rank,seo_review
3,client_23a62021009f63c4,content_559cdd76da9306de,3141.225806,36.712074,115320.914249,high_visibility_low_rank,seo_review
4,client_23a62021009f63c4,content_bdf60c86117079be,3626.741935,30.769353,111592.504282,high_visibility_low_rank,seo_review
5,client_23a62021009f63c4,content_ab91e088440ace78,2514.935484,43.166324,108560.519627,high_visibility_low_rank,seo_review
6,client_23a62021009f63c4,content_3df3f32f3fd58dea,4521.161290,23.335465,105503.400633,high_visibility_low_rank,seo_review
7,client_20259bd6705d81d4,content_82e35c4845e6c391,4642.161290,22.558608,104720.699038,high_visibility_low_rank,seo_review
8,client_23a62021009f63c4,content_df47d1b976106de4,4248.612903,24.355625,103477.622005,high_visibility_low_rank,seo_review
9,client_23a62021009f63c4,content_96e6613b42b52c42,2058.677419,46.053412,94809.120035,high_visibility_low_rank,seo_review


## 3. Top-10 review

The following review examines the ten highest-scoring pages from the baseline queue. Each recommendation is based only on the historical Search Console metrics used by the baseline rule.

| Rank | Action | Reason code | Confidence note | What would make it wrong |
|------|--------|-------------|-----------------|--------------------------|
| 1 | seo_review | high_visibility_low_rank | Medium confidence. The page scores highly based on historical impressions and average search position. | Rankings may already be improving after the March observation period. |
| 2 | seo_review | high_visibility_low_rank | Medium confidence. High impressions suggest strong search demand, while the average position indicates room for improvement. | Recent SEO changes may not yet be reflected in the available data. |
| 3 | seo_review | high_visibility_low_rank | Medium confidence. The rule consistently identifies this page as having high visibility with below-optimal rankings. | The page may already perform as expected for its search intent. |
| 4 | seo_review | high_visibility_low_rank | Medium confidence. Historical search metrics indicate a potential optimization opportunity. | Competition for target queries may limit ranking improvements. |
| 5 | seo_review | high_visibility_low_rank | Medium confidence. The page combines meaningful impressions with a relatively poor average position. | Business priorities may place this page below other optimization efforts. |
| 6 | seo_review | high_visibility_low_rank | Medium confidence. The score reflects sustained search visibility and ranking improvement potential. | Seasonal search behavior may have influenced the observed metrics. |
| 7 | seo_review | high_visibility_low_rank | Medium confidence. The rule prioritizes this page because both selected signals contribute to a high score. | Additional content or technical factors not included in the rule may explain its ranking. |
| 8 | seo_review | high_visibility_low_rank | Medium confidence. Historical impressions indicate existing visibility worth protecting and improving. | The page may already satisfy business objectives despite its ranking. |
| 9 | seo_review | high_visibility_low_rank | Medium confidence. The page has measurable search demand but does not consistently rank near the top results. | Search demand may decline independently of page quality. |
| 10 | seo_review | high_visibility_low_rank | Medium confidence. The combination of impressions and average position produces a strong baseline score. | External factors such as algorithm changes could influence rankings. |

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = baseline.head(10)

display(top10)

print(f"Reviewed {len(top10)} highest-scoring pages.")

,client_hash_id,content_hash_id,avg_impressions,avg_position,score,reason_code,action_label
0,client_23a62021009f63c4,content_36e53e9c707674fc,6276.741935,32.766674,205667.959346,high_visibility_low_rank,seo_review
1,client_23a62021009f63c4,content_e8a52cf3d5988c07,7901.000000,15.008339,118580.884331,high_visibility_low_rank,seo_review
2,client_23a62021009f63c4,content_73aa61dcedebbf30,2584.645161,45.700431,118119.397155,high_visibility_low_rank,seo_review
3,client_23a62021009f63c4,content_559cdd76da9306de,3141.225806,36.712074,115320.914249,high_visibility_low_rank,seo_review
4,client_23a62021009f63c4,content_bdf60c86117079be,3626.741935,30.769353,111592.504282,high_visibility_low_rank,seo_review
5,client_23a62021009f63c4,content_ab91e088440ace78,2514.935484,43.166324,108560.519627,high_visibility_low_rank,seo_review
6,client_23a62021009f63c4,content_3df3f32f3fd58dea,4521.161290,23.335465,105503.400633,high_visibility_low_rank,seo_review
7,client_20259bd6705d81d4,content_82e35c4845e6c391,4642.161290,22.558608,104720.699038,high_visibility_low_rank,seo_review
8,client_23a62021009f63c4,content_df47d1b976106de4,4248.612903,24.355625,103477.622005,high_visibility_low_rank,seo_review
9,client_23a62021009f63c4,content_96e6613b42b52c42,2058.677419,46.053412,94809.120035,high_visibility_low_rank,seo_review


Reviewed 10 highest-scoring pages.


## 4. Weak picks + leakage check

### Weak picks

This baseline uses only two historical Search Console signals: average impressions and average search position. As a result, some recommendations may not actually require SEO review.

Possible weak picks include:

- Pages whose rankings are already improving after the March 2026 observation period.
- Pages targeting highly competitive search queries where ranking improvements are difficult.
- Pages that already meet business objectives despite lower average search positions.

These limitations highlight that the baseline is intended as a transparent decision-support rule rather than a final decision system.

### Leakage check

The baseline score uses only historical Search Console metrics from the March 2026 partition.

No future reporting periods were used.

No FlyRank product flags or action labels were used as model inputs.

No label-derived variables or future outcomes were used to calculate the baseline score.

The rule is intended as an honest baseline for comparison with future machine learning models.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leakage_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS usable_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

display(leakage_check)

print("\nObservation:")
print("- The baseline was built using only rows with historical Search Console data available.")
print("- The analysis is limited to the March 2026 partition.")
print("- No future reporting periods were queried.")
print("- No FlyRank product flags or label-derived variables were used.")

,total_rows,usable_rows
0,9841378,3611061



Observation:
- The baseline was built using only rows with historical Search Console data available.
- The analysis is limited to the March 2026 partition.
- No future reporting periods were queried.
- No FlyRank product flags or label-derived variables were used.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`